<a href="https://colab.research.google.com/github/kevins-winter/multipose2/blob/main/notebooks/run_cellpose3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install and run multipose2 denoising and segmentation
## ⚠️ **Warning:**  this notebook uses the multipose2 denoising workflow that corresponds to the legacy cellpose3-style restoration models. Be careful with your environments and the `pip` command below. ⚠️

## This notebook uses the restoration models exposed through multipose2.denoise for denoising and segmentation workflows. 

# Running multipose2 in colab with a GPU

<font size = 4>multipose2 includes restoration models for noisy, blurry, and low-resolution images.

For more details on the restoration workflow, check the multipose2 `denoise` module and associated model docs.

Mount your google drive to access all your image files. This also ensures that the segmentations are saved to your google drive.

## Installation

Install multipose2. In Colab, the torch GPU build is typically already available.

## Notebook progress tracker

Use the next cell to track where you are in this notebook.

- Run `progress.show()` to refresh the current status.
- Run `progress.advance()` when you finish a phase.
- Run `progress.set_phase("...")` if you want to jump to a specific phase.
- Progress is saved to `multipose2_progress.json` on Google Drive if it is mounted, otherwise to `.multipose2_progress.json` in the current working directory.


In [ ]:
from pathlib import Path
import json
from datetime import datetime
from IPython.display import Markdown, display


class NotebookProgressTracker:
    def __init__(self, notebook_name, phases, state_path=None):
        self.notebook_name = notebook_name
        self.phases = list(phases)
        self.state_path = Path(state_path) if state_path else self._default_state_path()
        self.state = self._load_state()
        entry = self.state.setdefault(
            self.notebook_name,
            {"current_phase": self.phases[0], "updated_at": None},
        )
        if entry.get("current_phase") not in self.phases:
            entry["current_phase"] = self.phases[0]
        self._save()

    def _default_state_path(self):
        drive_path = Path('/content/drive/MyDrive')
        if drive_path.exists():
            return drive_path / 'multipose2_progress.json'
        return Path.cwd() / '.multipose2_progress.json'

    def _load_state(self):
        if self.state_path.exists():
            try:
                return json.loads(self.state_path.read_text())
            except json.JSONDecodeError:
                pass
        return {}

    def _save(self):
        self.state_path.parent.mkdir(parents=True, exist_ok=True)
        self.state_path.write_text(json.dumps(self.state, indent=2))

    def current_phase(self):
        return self.state[self.notebook_name]['current_phase']

    def current_index(self):
        return self.phases.index(self.current_phase())

    def set_phase(self, phase):
        if isinstance(phase, int):
            phase = self.phases[phase]
        if phase not in self.phases:
            raise ValueError(f'Unknown phase: {phase!r}')
        self.state[self.notebook_name] = {
            'current_phase': phase,
            'updated_at': datetime.now().isoformat(timespec='seconds'),
        }
        self._save()
        self.show()

    def advance(self):
        next_index = min(self.current_index() + 1, len(self.phases) - 1)
        self.set_phase(next_index)

    def rewind(self):
        prev_index = max(self.current_index() - 1, 0)
        self.set_phase(prev_index)

    def reset(self):
        self.set_phase(0)

    def show(self):
        current_index = self.current_index()
        updated_at = self.state[self.notebook_name].get('updated_at') or 'not saved yet'
        lines = [
            f'### Notebook Progress: {self.notebook_name}',
            f'- State file: `{self.state_path}`',
            f'- Last updated: `{updated_at}`',
            '',
        ]
        for index, phase in enumerate(self.phases):
            marker = 'x' if index < current_index else '>' if index == current_index else ' '
            lines.append(f'- [{marker}] {phase}')
        lines.extend([
            '',
            "Run `progress.advance()` when you finish the current phase.",
            "Use `progress.rewind()`, `progress.reset()`, or `progress.set_phase(...)` when needed.",
        ])
        display(Markdown('\\n'.join(lines)))


NOTEBOOK_PHASES = ['GPU and runtime setup', 'Install multipose2', 'Check CUDA and imports', 'Load images or mount Drive', 'Run denoising and segmentation', 'Review results']
progress = NotebookProgressTracker('run_cellpose3', NOTEBOOK_PHASES)
progress.show()


In [ ]:
!pip install "opencv-python-headless>=4.9.0.80"
!pip install git+https://github.com/kevins-winter/multipose2.git

Check CUDA version and that GPU is working in multipose2, then import other libraries.

In [ ]:
!nvcc --version
!nvidia-smi

import os, shutil
import numpy as np
import matplotlib.pyplot as plt
from multipose2 import core, utils, io, models, metrics
from glob import glob

use_GPU = core.use_gpu()
yn = ['NO', 'YES']
print(f'>>> GPU activated? {yn[use_GPU]}')

## Images

Load in your own data or use ours (below)

In [ ]:
import numpy as np
import time, os, sys
from urllib.parse import urlparse
import matplotlib.pyplot as plt
import matplotlib as mpl
%matplotlib inline
mpl.rcParams['figure.dpi'] = 200
from multipose2 import utils, io

# download noisy images from website
url = "http://www.cellpose.org/static/data/test_poisson.npz"
filename = "test_poisson.npz"
utils.download_url_to_file(url, filename)
dat = np.load(filename, allow_pickle=True)["arr_0"].item()

imgs = dat["test_noisy"]
plt.figure(figsize=(8,3))
for i, iex in enumerate([2, 18, 20]):
    img = imgs[iex].squeeze()
    plt.subplot(1,3,1+i)
    plt.imshow(img, cmap="gray", vmin=0, vmax=1)
    plt.axis('off')
plt.tight_layout()
plt.show()

Mount your google drive here if you want to load your own images:

In [ ]:

#@markdown ###Run this cell to connect your Google Drive to Colab

#@markdown * Click on the URL.

#@markdown * Sign in your Google Account.

#@markdown * Copy the authorization code.

#@markdown * Enter the authorization code.

#@markdown * Click on "Files" site on the right. Refresh the site. Your Google Drive folder should now be available here as "drive".

#mounts user's Google Drive to Google Colab.

from google.colab import drive
drive.mount('/content/gdrive')


## run denoising and segmentation

In [ ]:
# RUN CELLPOSE3

from multipose2 import denoise, io

io.logger_setup() # run this to get printing of progress

# DEFINE CELLPOSE MODEL
# model_type="cyto3" or "nuclei", or other model
# restore_type: "denoise_cyto3", "deblur_cyto3", "upsample_cyto3", "denoise_nuclei", "deblur_nuclei", "upsample_nuclei"
model = denoise.CellposeDenoiseModel(gpu=True, model_type="cyto3",
                                     restore_type="denoise_cyto3")

# define CHANNELS to run segementation on
# grayscale=0, R=1, G=2, B=3
# channels = [cytoplasm, nucleus]
# if NUCLEUS channel does not exist, set the second channel to 0
# channels = [0,0]
# IF ALL YOUR IMAGES ARE THE SAME TYPE, you can give a list with 2 elements
# channels = [0,0] # IF YOU HAVE GRAYSCALE
# channels = [2,3] # IF YOU HAVE G=cytoplasm and B=nucleus
# channels = [2,1] # IF YOU HAVE G=cytoplasm and R=nucleus
# OR if you have different types of channels in each image
# channels = [[2,3], [0,0], [0,0]]

# if you have a nuclear channel, you can use the nuclei restore model on the nuclear channel with
# model = denoise.CellposeDenoiseModel(..., chan2_restore=True)

# NEED TO SPECIFY DIAMETER OF OBJECTS
# in this case we have them from the ground-truth masks
diams = dat["diam_test"]

masks, flows, styles, imgs_dn = model.eval(imgs, diameter=diams, channels=[0,0])


plot results

In [ ]:
plt.figure(figsize=(8,12))
for i, iex in enumerate([2, 18, 20]):
    img = imgs[iex].squeeze()
    plt.subplot(3,3,1+i)
    plt.imshow(img, cmap="gray", vmin=0, vmax=1)
    plt.axis('off')
    plt.title("noisy")

    img_dn = imgs_dn[iex].squeeze()
    plt.subplot(3,3,4+i)
    plt.imshow(img_dn, cmap="gray", vmin=0, vmax=1)
    plt.axis('off')
    plt.title("denoised")

    plt.subplot(3,3,7+i)
    plt.imshow(img_dn, cmap="gray", vmin=0, vmax=1)
    outlines = utils.outlines_list(masks[iex])
    for o in outlines:
        plt.plot(o[:,0], o[:,1], color=[1,1,0])
    plt.axis('off')
    plt.title("segmentation")

plt.tight_layout()
plt.show()